# Inference V6
Pre-result + threshold application pipeline (app.anomaly_detection_v6).

In [ ]:
import os
import sys

# Add backend to path for app/... imports
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [ ]:
import json
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from pyspark.sql import functions as F

In [ ]:
import app.anomaly_detection_v6.constants as constants
from app.anomaly_detection_v6.utils import set_reproducibility
from app.anomaly_detection_v6.models import SupportedAnomalyDetectionModels
from app.anomaly_detection_v6.inference.preresult_inferencer import PreResultInferencer
from app.anomaly_detection_v6.inference.threshold_applier import ThresholdApplier
from app.anomaly_detection_v6.inference.utils import analyze_trigger_sources, visualize_feature_level_report_vq

In [ ]:
set_reproducibility(seed=constants.RANDOM_SEED)

MODEL_DIR = "/Volumes/dbxws_multimodalai001/bronze_layer/file/anomaly_detection_experiments/20260406-Exp/checkpoints"
MODEL_NAME = "robot_anomaly_model_rvq"
WEIGHTS_FILE = f"model_{MODEL_NAME}_best.pt"
METADATA_FILE = f"metadata_{MODEL_NAME}_best.json"

QUANTILE = 0.95
K_SKIP = 1

TEST_TABLES = [
    "dbxws_multimodalai001.silver_layer.slicingwindow_data_variable_robot_indexed_test",
]

PRE_RESULT_TABLES = [f"{t}_pre_res_v6" for t in TEST_TABLES]
RESULT_TABLES = [f"{t}_res_v6" for t in TEST_TABLES]

In [ ]:
metadata_path = os.path.join(MODEL_DIR, METADATA_FILE)
weights_path = os.path.join(MODEL_DIR, WEIGHTS_FILE)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

state_dict = torch.load(weights_path, map_location="cpu")

metadata_br = spark.sparkContext.broadcast(metadata)
state_dict_br = spark.sparkContext.broadcast(state_dict)

In [ ]:
def ensure_window_time_col(df):
    if "window_start_time" not in df.columns:
        return df.withColumn("window_start_time", F.col("start_time"))
    return df

pre_result_inferencer = PreResultInferencer(
    model_version=SupportedAnomalyDetectionModels.V6_RVQ,
    metadata_broadcast=metadata_br,
    state_dict_broadcast=state_dict_br,
    quantile=QUANTILE,
    k_skip=K_SKIP,
)

pre_result_fn = pre_result_inferencer.make_inference_fn()

for src_table, pre_table in zip(TEST_TABLES, PRE_RESULT_TABLES):
    src_df = ensure_window_time_col(spark.table(src_table))
    pre_df = (
        src_df.groupBy("robot_code", "program_number", "start_time")
        .applyInPandas(pre_result_fn, schema=pre_result_inferencer.schema)
    )
    pre_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(pre_table)

# Threshold configuration

In [ ]:
threshold_config_path = os.path.join(MODEL_DIR, "threshold_config_v6.json")

if os.path.exists(threshold_config_path):
    with open(threshold_config_path, "r", encoding="utf-8") as f:
        threshold_config = json.load(f)
else:
    threshold_config = {sensor: {"shape": 5.0, "win": 0.5} for sensor in constants.SENSORS}

threshold_config_br = spark.sparkContext.broadcast(threshold_config)

In [ ]:
threshold_applier = ThresholdApplier(threshold_config_br, quantile=QUANTILE)
apply_thresholds_fn = threshold_applier.make_apply_fn()

for pre_table, res_table in zip(PRE_RESULT_TABLES, RESULT_TABLES):
    pre_df = spark.table(pre_table)
    res_df = (
        pre_df.groupBy("robot_code", "program_number", "start_time")
        .applyInPandas(apply_thresholds_fn, schema=threshold_applier.schema)
    )
    res_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(res_table)

In [ ]:
for res_table in RESULT_TABLES:
    print(res_table, spark.table(res_table).count())

# Step 3: Analyze trigger sources (shape vs win)

In [ ]:
for res_table in RESULT_TABLES:
    if spark.catalog.tableExists(res_table):
        analyze_trigger_sources(spark, res_table)
    else:
        print(f"Missing table: {res_table}")

# Visualization helper

In [ ]:
EXAMPLE_VISUALIZE = False
if EXAMPLE_VISUALIZE and RESULT_TABLES:
    if spark.catalog.tableExists(RESULT_TABLES[0]):
        sample_pdf = spark.table(RESULT_TABLES[0]).limit(1).toPandas()
        if len(sample_pdf) > 0:
            visualize_feature_level_report_vq(
                spark,
                sample_pdf.iloc[0],
                TEST_TABLES[0],
                k_skip=K_SKIP
            )
    else:
        print(f"Missing table: {RESULT_TABLES[0]}")